# Milestone 3 – Data Acquisition, Validation & Preparation
## AI-Based Story Point Estimation for Agile Software Development

This notebook demonstrates all Milestone 3 requirements:
1. **Data Ingestion** – raw CSV load via ZenML step (TFX ExampleGen style)
2. **Schema Definition & Data Validation** – TFDV statistics, anomaly detection & fix
3. **Preprocessing & Feature Engineering** – text cleaning, feature creation
4. **Feature Store** – Feast registration
5. **Data Versioning** – DVC commands
6. **Pipeline Execution** – ZenML full pipeline run

---
## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import os
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TFDV
import tensorflow_data_validation as tfdv

# Pipeline steps
from ingestion  import ingest_data, _normalize_columns
from validation import validate_data
from transform  import preprocess_and_engineer, clean_text, build_input_text

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup complete.')

---
## 1. Data Ingestion

In [ ]:
# ── Load raw CSV ─────────────────────────────────────────────────────────────
CSV_PATH = '../data/raw/raw_data.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df_raw):,} rows and {len(df_raw.columns)} columns')
print(f'Columns: {list(df_raw.columns)}')

In [ ]:
# Normalize column names to canonical schema
df_raw = _normalize_columns(df_raw)
print('Normalized columns:', list(df_raw.columns))
df_raw.head(3)

In [ ]:
# Basic data quality checks
print('=== Null counts ===')
print(df_raw.isnull().sum())

print('\n=== Story points distribution ===')
print(df_raw['storypoints'].describe())

In [ ]:
# Filter invalid rows (null or negative story points)
df_raw = df_raw.dropna(subset=['storypoints'])
df_raw = df_raw[df_raw['storypoints'] > 0].reset_index(drop=True)
print(f'After filtering: {len(df_raw):,} rows')

In [ ]:
# Story point distribution visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_raw['storypoints'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Story Points Distribution (raw)')
axes[0].set_xlabel('Story Points')
axes[0].set_ylabel('Count')

top_vals = df_raw['storypoints'].value_counts().head(15)
axes[1].bar(top_vals.index.astype(str), top_vals.values, color='coral')
axes[1].set_title('Top 15 Most Frequent Story Point Values')
axes[1].set_xlabel('Story Points')
axes[1].set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../tfdv_output/storypoints_distribution.png', dpi=150)
plt.show()

---
## 2. Schema Definition & Data Validation (TFDV)

In [ ]:
# ── Train / eval split for TFDV ──────────────────────────────────────────────
train_df = df_raw.sample(frac=0.8, random_state=42)
eval_df  = df_raw.drop(train_df.index)
print(f'Train: {len(train_df):,} | Eval: {len(eval_df):,}')

In [ ]:
# ── Compute statistics ───────────────────────────────────────────────────────
os.makedirs('../tfdv_output', exist_ok=True)

train_stats = tfdv.generate_statistics_from_dataframe(train_df)
eval_stats  = tfdv.generate_statistics_from_dataframe(eval_df)
print('Statistics computed for train and eval splits.')

In [ ]:
# ── Visualise statistics ─────────────────────────────────────────────────────
# Compare train vs eval statistics side by side
tfdv.visualize_statistics(
    lhs_statistics=train_stats,
    rhs_statistics=eval_stats,
    lhs_name='TRAIN',
    rhs_name='EVAL'
)

In [ ]:
# ── Infer schema from training statistics ────────────────────────────────────
os.makedirs('../schema', exist_ok=True)
schema = tfdv.infer_schema(statistics=train_stats)
tfdv.display_schema(schema)
print('\nSchema inferred from training data.')

In [ ]:
# ── Check for anomalies in evaluation set ────────────────────────────────────
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)
tfdv.display_anomalies(anomalies)

if not anomalies.anomaly_info:
    print('✅ No anomalies detected in the evaluation set.')
else:
    print(f'⚠️  {len(anomalies.anomaly_info)} anomaly/anomalies detected (see above).')

In [ ]:
# ── Fix anomalies: relax schema constraints ──────────────────────────────────
# 1. Relax float domain on storypoints to accommodate wider eval range
for feature in schema.feature:
    if feature.name == 'storypoints':
        if feature.HasField('float_domain'):
            feature.float_domain.min = 0.0
            feature.float_domain.max = 200.0
    # 2. Allow up to 50% missing values in text fields
    if feature.name in ('title', 'description'):
        feature.presence.min_fraction = 0.5

# Re-validate with revised schema
revised_anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)
tfdv.display_anomalies(revised_anomalies)

if not revised_anomalies.anomaly_info:
    print('✅ All anomalies resolved after schema revision.')
else:
    print(f'⚠️  Remaining anomalies: {len(revised_anomalies.anomaly_info)}')

In [ ]:
# ── Save revised schema ───────────────────────────────────────────────────────
tfdv.write_schema_text(schema, '../schema/schema.pbtxt')
print('✅ Revised schema saved to schema/schema.pbtxt')

---
## 3. Preprocessing & Feature Engineering

In [ ]:
from transform import engineer_features, clean_text, build_input_text

# Apply full feature engineering pipeline
df_processed = engineer_features(df_raw.copy())
print(f'Processed shape: {df_processed.shape}')
df_processed[['input_text', 'storypoints', 'log_storypoints',
              'text_length', 'word_count', 'has_description',
              'is_fibonacci']].head(5)

In [ ]:
# Text cleaning demo
raw_example = df_raw['title'].iloc[0] + ' ' + str(df_raw['description'].iloc[0])
print('=== Raw text (first 300 chars) ===')
print(raw_example[:300])
print('\n=== Cleaned text ===')
print(clean_text(raw_example)[:300])

In [ ]:
# Feature distribution visualisation
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].hist(df_processed['text_length'], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Input Text Length Distribution')
axes[0,0].set_xlabel('Character count')

axes[0,1].hist(df_processed['word_count'], bins=40, color='coral', edgecolor='white')
axes[0,1].set_title('Word Count Distribution')
axes[0,1].set_xlabel('Words')

axes[1,0].hist(df_processed['log_storypoints'], bins=30, color='green', edgecolor='white')
axes[1,0].set_title('log1p(Story Points) Distribution')
axes[1,0].set_xlabel('log1p(story points)')

fibonacci_counts = df_processed['is_fibonacci'].value_counts()
axes[1,1].bar(['Non-Fibonacci', 'Fibonacci'], fibonacci_counts.values, color=['#e74c3c','#2ecc71'])
axes[1,1].set_title('Fibonacci vs Non-Fibonacci Story Points')

plt.tight_layout()
plt.savefig('../tfdv_output/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Save processed data as Parquet
import os
os.makedirs('../data/processed', exist_ok=True)
df_processed.to_parquet('../data/processed/processed_data.parquet', index=False)
print('✅ Processed data saved to data/processed/processed_data.parquet')
print(f'Parquet size: {os.path.getsize("../data/processed/processed_data.parquet") / 1024:.1f} KB')

---
## 4. Feature Store (Feast)

In [ ]:
# Show Feast feature view definition
feast_def_path = '../feast_repo/feature_repo/features.py'
if os.path.exists(feast_def_path):
    with open(feast_def_path) as f:
        print(f.read())
else:
    print('Run the ZenML pipeline first to generate Feast files.')

In [ ]:
# Apply Feast feature store (if feast CLI is installed)
import subprocess
result = subprocess.run(
    ['feast', 'apply'],
    cwd='../feast_repo/feature_repo',
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

---
## 5. Data Versioning (DVC)

In [ ]:
# DVC commands to run in terminal (shown here for documentation)
dvc_commands = """
# Initialize DVC (first time only)
dvc init

# Track raw data
dvc add data/raw/raw_data.csv
git add data/raw/raw_data.csv.dvc data/raw/.gitignore
git commit -m "feat: track raw data with DVC [milestone3]"
git tag -a v1.0-raw -m "Milestone 3: initial raw data"

# Track processed data
dvc add data/processed/processed_data.parquet
git add data/processed/processed_data.parquet.dvc data/processed/.gitignore
git commit -m "feat: add processed feature-engineered data [milestone3]"
git tag -a v1.0-processed -m "Milestone 3: processed data after feature engineering"

# Track TFDV schema
dvc add schema/schema.pbtxt
git add schema/schema.pbtxt.dvc
git commit -m "feat: add TFDV inferred schema [milestone3]"

# Push to remote (configure a remote first: dvc remote add -d myremote gdrive://...)
dvc push
"""
print(dvc_commands)

---
## 6. Full ZenML Pipeline Run

In [ ]:
# Run the complete pipeline programmatically
import sys
sys.path.insert(0, '../pipeline')
from zenml_pipeline import data_pipeline

data_pipeline(csv_path='../data/raw/raw_data.csv')
print('✅ Full ZenML pipeline complete.')

---
## Summary

| Requirement | Status | Evidence |
|---|---|---|
| Data ingestion & raw storage | ✅ | `data/raw/raw_data.csv` (DVC tracked) |
| Schema definition | ✅ | `schema/schema.pbtxt` (TFDV inferred) |
| Data validation & anomaly detection | ✅ | `tfdv_output/anomalies_report.json` |
| Preprocessing & feature engineering | ✅ | `data/processed/processed_data.parquet` |
| Feature store | ✅ | Feast: `feast_repo/feature_repo/features.py` |
| Data versioning | ✅ | DVC `.dvc` files + git tags |
| ML pipeline integration | ✅ | ZenML `data_pipeline` in `pipeline/zenml_pipeline.py` |